In [2]:
import math
import random

In [3]:
# implementing micregrad frem scratch: https://yeutu.be/VMj-3S1tku0

class Value:
    def __init__(self, data, _children = (), _op='', label = ''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out


    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward

        return out

    def __radd__(self, other):
        return self + other

    def __rmul__(self, other):
        return self * other

    def __truediv___(self, other):
        return self * other**-1

    def __neg__(self):
        return self * -1

    def __sub__(self, other):
        return self + (-other)

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only supporting int/float powers for now"
        out = Value(self.data**other, (self,), f'**{other}')

        def _backward():
            self.grad += (other * (self.data **(other - 1))) * out.grad

        out._backward = _backward

        return out

    def exp(self):
        x = self.data
        out = Value(math.exp(x), (self,), 'exp')

        def _backward():
            self.grad += out.data * out.grad

        out._backward = _backward
        return out

    def tanh(self):
        x= self.data
        t = (math.exp(2*x) -1)/(math.exp(2*x)+1)
        out = Value(t, (self,), 'tanh')

        def _backward():
            self.grad += (1 - t**2)*out.grad

        out._backward = _backward
        return out

    def backward(self):
        # Using topological sort

        topo = []

        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)

        self.grad = 1.0
        for node in reversed(topo):
            node._backward()


print("Done")

Done


In [4]:
# Inputs

x1 = Value(2.0, label='x1')
x2 = Value(0.0, label='x2')

# Weights
w1 = Value(-3.0, label='w1')
w2 = Value(1.0, label='w2')

# Bias
b = Value(6.8813735870195432, label='b')

x1w1 = x1 * w1; x1w1.label = 'x1*w1'
x2w2 = x2 * w2; x2w2.label = 'x2*w2'

# Weighted Sum
x1w1x2w2 = x1w1 + x2w2; x1w1x2w2.label = 'x1*w1 + x2*w2'
n = x1w1x2w2 + b; n.label = 'n'

# Output
o = n.tanh(); o.label = 'o'

# Backpropagation
o.backward()

In [5]:
# Inputs

x1 = Value(2.0, label='x1')
x2 = Value(0.0, label='x2')

# Weights
w1 = Value(-3.0, label='w1')
w2 = Value(1.0, label='w2')

# Bias
b = Value(6.8813735870195432, label='b')

x1w1 = x1 * w1; x1w1.label = 'x1*w1'
x2w2 = x2 * w2; x2w2.label = 'x2*w2'

# Weighted Sum
x1w1x2w2 = x1w1 + x2w2; x1w1x2w2.label = 'x1*w1 + x2*w2'
n = x1w1x2w2 + b; n.label = 'n'

# Output
e = (2*n).exp(); e.label = 'e'
o = (e-1)*(e+1)**-1; o.label = 'o'

# Backpropagation
o.backward()

In [6]:
# Using Pytorch to verify the results


import torch

In [7]:
x1 = torch.tensor(2.0, requires_grad=True, dtype=torch.float64)
x2 = torch.tensor(0.0, requires_grad=True, dtype=torch.float64)



w1 = torch.tensor(-3.0, requires_grad=True, dtype=torch.float64)
w2 = torch.tensor(1.0, requires_grad=True, dtype=torch.float64)

b = torch.tensor(6.8813735870195432, requires_grad=True, dtype=torch.float64)

n = x1*w1 + x2*w2 + b

o = torch.tanh(n)

o.backward()

# # print(f"PyTorch output: {o.item()}")
# print('b.grad: ', b.grad.item())

# print("--------------------------------")
#     print('x1.grad: ', x1.item())
#     print('x2.grad: ', x2.grad.item())


In [8]:
class Neuron:
    """A mathematical representation of a single neuron in a neural network."""
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(0)

    def __call__(self, x):
        act = sum((wi*xi for wi, xi in zip(self.w, x)), self.b)
        out = act.tanh()
        return out

    def parameters(self):
        return self.w + [self.b]


class Layer:
    """A layer of neurons in a neural network: a list of neurons that take the same input and produce an output."""
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs

    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()]


class MLP: # MLP for `Multi-Layer Perceptron`
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x= layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]


x= [2.0, 3.0, -1.0]


n = MLP(3, [4, 4, 1])
n(x)

Value(data=0.6172036482341469)

In [9]:
# Going into the main events

xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0],
]

ys = [1.0, -1.0, -1.0, 1.0] # desired targets



In [10]:
for k in range(20):

    # -------> Forward Pass <-------
    ypred= [n(x) for x in xs]
    loss = sum((yout - ygt)**2 for yout, ygt in zip(ypred, ys)) # Mean Square Error Method here

    # |------->> Backward Pass <<-------|

    #       |---* Zero Grad *---|
    for p in n.parameters():
        p.grad = 0.0

    loss.backward()

    # Update
    for p in n.parameters():
        p.data += - 0.05 * p.grad # stochastic gradient descent

    print(f"Epoch {k+1} --- loss {loss.data}")

Epoch 1 --- loss 0.29249090327150173
Epoch 2 --- loss 0.16572101003715567
Epoch 3 --- loss 0.134093841014382
Epoch 4 --- loss 0.11328319872227141
Epoch 5 --- loss 0.09812589720308815
Epoch 6 --- loss 0.08650762023114665
Epoch 7 --- loss 0.07729740122329491
Epoch 8 --- loss 0.06981199919868769
Epoch 9 --- loss 0.06360806197263362
Epoch 10 --- loss 0.05838342828938732
Epoch 11 --- loss 0.0539244126326412
Epoch 12 --- loss 0.050075388070759834
Epoch 13 --- loss 0.04672022815268008
Epoch 14 --- loss 0.04377048591774384
Epoch 15 --- loss 0.04115759756158521
Epoch 16 --- loss 0.03882758899609672
Epoch 17 --- loss 0.03673739160854781
Epoch 18 --- loss 0.034852222278288866
Epoch 19 --- loss 0.03314368465839368
Epoch 20 --- loss 0.03158836985971979


In [11]:
ypred

[Value(data=0.911732710334065),
 Value(data=-0.901557084930954),
 Value(data=-0.9175215915958482),
 Value(data=0.9145391314426523)]